# Hazelnut Price Regression: Production + TMO → Giresun Price

Source of truth: `giresun_spot_prices_monthly.csv` (Giresun exchange).

**Two models:**
1. **Monthly** — predict monthly exchange price from annual production, crop-year TMO regime, and seasonality
2. **Annual** — predict crop-year VWAP from annual production and TMO

**Targets:** TRY/kg and USD/kg. TRY is dominated by Turkish inflation so returns are more informative than levels for TRY; USD levels work directly.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')

ROOT = Path('..')
DATA = ROOT / 'data' / 'raw'

## 1. Load Data

In [ ]:
# Monthly Giresun exchange data — source of truth
monthly = pd.read_csv(DATA / 'giresun_spot_prices_monthly.csv')
print(f'Monthly: {monthly.shape}  ({monthly.crop_year.min()}–{monthly.crop_year.max()})')
monthly.head(3)

In [ ]:
# Master: annual production + TMO floor (corrected from exchange data)
master = pd.read_csv(DATA / 'hazelnut_35yr_master.csv', index_col=0)
master.index = master.index.astype(int)
print(f'Master: {master.shape}  ({master.index.min()}–{master.index.max()})')
master[['prod_mt', 'prod_shortfall', 'tmo_try', 'tmo_active']].tail(10)

## 2. Feature Engineering

In [ ]:
# Trailing 5yr production trend — backward-only, no look-ahead
# shift(1) ensures year t uses mean of t-5..t-1 only
master['trend_trail5']   = master['prod_mt'].rolling(5, min_periods=3).mean().shift(1)
master['shortfall_trail'] = (master['prod_mt'] - master['trend_trail5']) / master['trend_trail5'] * 100

# Join crop-year production and TMO onto monthly rows
prod_cols = master[['prod_mt', 'prod_shortfall', 'shortfall_trail', 'tmo_try', 'tmo_active', 'prod_trend_5yr']].copy()
prod_cols.index.name = 'crop_year'

df = monthly.merge(prod_cols, on='crop_year', how='left')

# Sequential crop month (1 = first month of sales for that crop year)
df['crop_month'] = df.groupby('crop_year').cumcount() + 1

# Log price targets
df['log_try'] = np.log(df['avg_try_kg_inshell'].replace(0, np.nan))
df['log_usd'] = np.log(df['avg_usd_kg_inshell'].replace(0, np.nan))

# Log production features
df['log_prod']        = np.log(df['prod_mt'])
df['shortfall_pct']   = df['prod_shortfall'] * 100
df['log_tmo']         = np.log(df['tmo_try'].replace(0, np.nan))
df['tmo_active_flag'] = df['tmo_active'].fillna(0).astype(int)

# Month-over-month log return (for TRY where levels are inflation-dominated)
df = df.sort_values(['crop_year', 'crop_month'])
df['ret_try'] = df.groupby('crop_year')['log_try'].diff()   # within crop year
df['ret_usd'] = df.groupby('crop_year')['log_usd'].diff()

# AR(1): lagged log price (cross-crop-year, for level model)
df_sorted = df.sort_values(['crop_year', 'crop_month'])
df['lag_log_try'] = df_sorted['log_try'].shift(1).values
df['lag_log_usd'] = df_sorted['log_usd'].shift(1).values

print(f'Panel: {len(df)} rows, {df.crop_year.nunique()} crop years')
print(f'Trailing shortfall coverage: {df["shortfall_trail"].notna().sum()} rows (NaN in first ~5 crop years)')
print(f'\nNulls in key columns:')
print(df[['log_try','log_usd','log_prod','shortfall_pct','shortfall_trail','tmo_try','tmo_active_flag']].isnull().sum())

## 3. Exploratory Plots

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=False)

# Price time series
ax = axes[0]
ax.plot(df.index, df['avg_try_kg_inshell'], lw=0.8, color='#2c3e50', label='TRY/kg')
ax.set_yscale('log')
ax.set_ylabel('TRY/kg (log scale)')
ax.set_title('Monthly Giresun exchange price — TRY/kg (log scale)')

# USD price
ax = axes[1]
ax2 = ax.twinx()
ax.plot(df.index, df['avg_usd_kg_inshell'], lw=0.8, color='steelblue', label='USD/kg')
# shade TMO-binding crop years
tmo_yrs = df[df['tmo_active_flag'] == 1]['crop_year'].unique()
for yr in tmo_yrs:
    sub = df[df['crop_year'] == yr]
    ax.axvspan(sub.index.min(), sub.index.max(), alpha=0.08, color='red')
ax.set_ylabel('USD/kg')
ax.set_title('USD/kg — red shading = TMO binding crop years')

# Production by crop year
ax = axes[2]
cy_prod = df.groupby('crop_year')['prod_mt'].first()
ax.bar(cy_prod.index, cy_prod.values / 1e3, color='#bdc3c7', label='Production (kt)')
trend = df.groupby('crop_year')['prod_trend_5yr'].first()
ax.plot(trend.index, trend.values / 1e3, 'k--', lw=1, label='5yr trend')
ax.set_ylabel('000 tonnes')
ax.set_xlabel('Crop year')
ax.legend(fontsize=8)
ax.set_title('Annual production by crop year')

plt.tight_layout()
plt.show()

In [ ]:
# Within-year price seasonality by crop month
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, col, label in [
    (axes[0], 'log_try', 'log(TRY/kg)'),
    (axes[1], 'log_usd', 'log(USD/kg)'),
]:
    monthly_med = df.groupby('crop_month')[col].median()
    monthly_q25 = df.groupby('crop_month')[col].quantile(0.25)
    monthly_q75 = df.groupby('crop_month')[col].quantile(0.75)
    ax.plot(monthly_med.index, monthly_med.values, 'k-', lw=1.5, label='Median')
    ax.fill_between(monthly_med.index, monthly_q25, monthly_q75, alpha=0.2, color='steelblue', label='IQR')
    ax.set_xlabel('Crop month (1 = first month of exchange sales)')
    ax.set_ylabel(label)
    ax.set_title(f'Within-year seasonality: {label}')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 4. Monthly Regression

**Note on look-ahead bias:** Annual production is not confirmed until September/October harvest announcement. Monthly prices in August trade on *expectations*, not the realized shortfall number. Using realized `shortfall_pct` as a regressor for all months within a crop year is fine for historical description but would break in a real-time forecasting context.

**Note on `tmo_active`:** Dropped from all models. TMO buys when the market is already weak (oversupply), making it mechanically correlated with the production channel — `tmo_active=1` is not an independent cause of low prices, it's a consequence of the same surplus that drives `shortfall_pct` positive. Including it as a regressor just adds a correlated proxy that soaks up variance from the variable you actually care about.

### 4a. Monthly USD (log-level, no lag) — production signal after stripping seasonality

In [ ]:
# Monthly USD level model — with lag price (high R² but dominated by AR persistence)
m_usd = df.dropna(subset=['log_usd', 'log_prod', 'lag_log_usd']).copy()
m_usd['crop_month_sq'] = m_usd['crop_month'] ** 2

X_usd = sm.add_constant(m_usd[['log_prod', 'crop_month', 'crop_month_sq', 'lag_log_usd']])
model_usd = sm.OLS(m_usd['log_usd'], X_usd).fit(
    cov_type='cluster', cov_kwds={'groups': m_usd['crop_year']})
print(model_usd.summary())

In [ ]:
# No lag — honest production signal after controlling for seasonality only
# R² here is the defensible number: how much does production explain month-to-month prices?
m_usd2 = df.dropna(subset=['log_usd', 'shortfall_pct']).copy()
m_usd2['crop_month_sq'] = m_usd2['crop_month'] ** 2

X_simple = sm.add_constant(m_usd2[['shortfall_pct', 'crop_month', 'crop_month_sq']])
model_usd_simple = sm.OLS(m_usd2['log_usd'], X_simple).fit(
    cov_type='cluster', cov_kwds={'groups': m_usd2['crop_year']})
print(model_usd_simple.summary())

### 4b. TRY price (within-year log-return) — strips inflation

### 4b. TRY price (log-return) — strips inflation

In [ ]:
m_try = df.dropna(subset=['ret_try', 'shortfall_pct']).copy()
m_try = m_try[m_try['crop_month'] > 1]
m_try['crop_month_sq'] = m_try['crop_month'] ** 2

X_try = sm.add_constant(m_try[['shortfall_pct', 'crop_month', 'crop_month_sq']])
model_try = sm.OLS(m_try['ret_try'], X_try).fit(
    cov_type='cluster', cov_kwds={'groups': m_try['crop_year']})
print(model_try.summary())

### 4c. With TMO floor price (where available — n=12 crop years)

This is NOT endogenous: `log(tmo_try)` is the announced floor price, not a regime dummy. The question is whether, conditional on the announced floor, production shortfall still has incremental explanatory power for the actual market price.

In [ ]:
m_tmo = df.dropna(subset=['log_usd', 'log_prod', 'log_tmo', 'lag_log_usd']).copy()
m_tmo['crop_month_sq'] = m_tmo['crop_month'] ** 2
print(f'Sample with TMO prices: n={len(m_tmo)} rows, {m_tmo.crop_year.nunique()} crop years')

X_tmo = sm.add_constant(m_tmo[['log_prod', 'log_tmo', 'crop_month', 'crop_month_sq', 'lag_log_usd']])
model_tmo = sm.OLS(m_tmo['log_usd'], X_tmo).fit(
    cov_type='cluster', cov_kwds={'groups': m_tmo['crop_year']})
print(model_tmo.summary())

## 5. Annual Regression (crop-year level) — primary model

**Best model for insurance pricing:** `ret_vwap_usd ~ shortfall_pct`

- Log-returns remove TRY/USD secular drift that corrupts levels
- `shortfall_pct` is the causal variable (frost → volume loss → price spike)
- Annual aggregation avoids the look-ahead and within-year autocorrelation problems
- Coefficient is directly interpretable: 10% below-trend production → X% expected price increase
- `tmo_active` dropped (endogenous — see note in Section 4)

In [ ]:
# Load prebuilt crop-year VWAP (correctly handles FX for all years)
cy = pd.read_csv(DATA / 'giresun_spot_prices_cropyear.csv').set_index('crop_year')

# Join production + TMO from master (trailing shortfall already computed in Section 2)
prod_cols = master[['prod_mt', 'prod_shortfall', 'shortfall_trail', 'tmo_try', 'tmo_active', 'prod_trend_5yr']].copy()
prod_cols.index.name = 'crop_year'
annual = cy.join(prod_cols, how='left')

annual['log_vwap_try']  = np.log(annual['vwap_try_kg_inshell'])
annual['log_vwap_usd']  = np.log(annual['vwap_usd_kg_inshell'])
annual['log_prod']      = np.log(annual['prod_mt'])
annual['log_tmo']       = np.log(annual['tmo_try'].replace(0, np.nan))
annual['shortfall_pct'] = annual['prod_shortfall'] * 100
annual['tmo_active']    = annual['tmo_active'].fillna(0).astype(int)
annual['ret_vwap_try']  = annual['log_vwap_try'].diff()
annual['ret_vwap_usd']  = annual['log_vwap_usd'].diff()
annual['prod_ret']      = annual['log_prod'].diff()
annual['prod_ret_lag1'] = annual['prod_ret'].shift(1)

print(f'Annual panel: {len(annual)} crop years  ({annual.index.min()}–{annual.index.max()})')
annual[['vwap_try_kg_inshell', 'vwap_usd_kg_inshell', 'prod_mt',
        'shortfall_pct', 'shortfall_trail', 'tmo_active', 'tmo_try']].round(2)

In [ ]:
# PRIMARY MODEL: annual USD return ~ TRAILING 5yr production shortfall (no look-ahead)
a3 = annual.dropna(subset=['ret_vwap_usd', 'shortfall_trail'])
res_a3 = sm.OLS(a3['ret_vwap_usd'], sm.add_constant(a3[['shortfall_trail']])).fit()
print(res_a3.summary())

# Comparison: centered shortfall (uses t+1,t+2 — look-ahead contaminated, inflated upper bound)
a3c = annual.dropna(subset=['ret_vwap_usd', 'shortfall_pct'])
res_a3c = sm.OLS(a3c['ret_vwap_usd'], sm.add_constant(a3c[['shortfall_pct']])).fit()
print(f'\nCentered 5yr trend  R²={res_a3c.rsquared:.3f}  (look-ahead bias — upper bound)')
print(f'Trailing 5yr trend  R²={res_a3.rsquared:.3f}  (clean — use this for pricing)')

In [ ]:
# Alternative: production log-return instead of shortfall_pct
a2 = annual.dropna(subset=['ret_vwap_usd', 'prod_ret'])
res_a2 = sm.OLS(a2['ret_vwap_usd'], sm.add_constant(a2[['prod_ret']])).fit()
print(res_a2.summary())

In [ ]:
# Log-levels instead of returns — for completeness, but note time-trend contamination
a1 = annual.dropna(subset=['log_vwap_usd', 'shortfall_pct'])
res_a1 = sm.OLS(a1['log_vwap_usd'], sm.add_constant(a1[['shortfall_pct']])).fit()
print(res_a1.summary())

In [ ]:
# With explicit TMO floor price (n=12, not endogenous — it's the announced level not a regime flag)
a4 = annual.dropna(subset=['ret_vwap_usd', 'shortfall_pct', 'log_tmo'])
print(f'With TMO floor: n={len(a4)} crop years')
res_a4 = sm.OLS(a4['ret_vwap_usd'], sm.add_constant(a4[['shortfall_pct', 'log_tmo']])).fit()
print(res_a4.summary())

## 6. Results Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: scatter + regression line
ax = axes[0]
ax.scatter(a3['shortfall_trail'], a3['ret_vwap_usd'] * 100, c='steelblue', s=50, zorder=3)
xs = np.linspace(a3['shortfall_trail'].min(), a3['shortfall_trail'].max(), 100)
coef = res_a3.params
ax.plot(xs, (coef['const'] + coef['shortfall_trail'] * xs) * 100, 'k--', lw=1.2)
for yr, row in a3.iterrows():
    if abs(row['shortfall_trail']) > 12 or abs(row['ret_vwap_usd']) > 0.3:
        ax.annotate(str(yr), (row['shortfall_trail'], row['ret_vwap_usd'] * 100),
                    xytext=(4, 3), textcoords='offset points', fontsize=7)
ax.axhline(0, color='grey', lw=0.5, ls='--')
ax.axvline(0, color='grey', lw=0.5, ls='--')
ax.set_xlabel('Production shortfall vs trailing 5yr trend (%)')
ax.set_ylabel('Annual USD price return (%)')
ax.set_title(
    f'PRIMARY MODEL  R²={res_a3.rsquared:.3f}\n'
    f'coef={coef["shortfall_trail"]*100:.2f}% per 1% shortfall  p={res_a3.pvalues["shortfall_trail"]:.3f}'
)

# Right: actual vs fitted
ax = axes[1]
fitted = res_a3.fittedvalues * 100
actual = a3['ret_vwap_usd'] * 100
ax.scatter(fitted, actual, c='steelblue', s=40, alpha=0.8)
for yr, (f, a_) in zip(a3.index, zip(fitted, actual)):
    if abs(a_ - f) > 15:
        ax.annotate(str(yr), (f, a_), xytext=(4, 3), textcoords='offset points', fontsize=7)
mn, mx = min(fitted.min(), actual.min()) - 2, max(fitted.max(), actual.max()) + 2
ax.plot([mn, mx], [mn, mx], 'k--', lw=0.8)
ax.set_xlabel('Fitted USD return (%)')
ax.set_ylabel('Actual USD return (%)')
ax.set_title(f'Annual model: actual vs fitted  (R²={res_a3.rsquared:.3f})')

plt.tight_layout()
plt.show()

In [ ]:
# Monthly USD model: actual vs fitted + residuals
fitted_m = model_usd_simple.fittedvalues
actual_m = m_usd2['log_usd']
resid_m  = actual_m - fitted_m

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].scatter(fitted_m, actual_m, s=8, alpha=0.5, c='steelblue')
mn, mx = min(fitted_m.min(), actual_m.min()), max(fitted_m.max(), actual_m.max())
axes[0].plot([mn, mx], [mn, mx], 'k--', lw=0.8)
axes[0].set_xlabel('Fitted log(USD/kg)'); axes[0].set_ylabel('Actual')
axes[0].set_title(f'Monthly USD model (no lag) — R²={model_usd_simple.rsquared:.3f}')

axes[1].hist(resid_m, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='red', lw=0.8, ls='--')
axes[1].set_xlabel('Residual'); axes[1].set_ylabel('Count')
axes[1].set_title(f'Residuals  σ={resid_m.std():.3f}')

axes[2].scatter(m_usd2['shortfall_pct'], resid_m, s=8, alpha=0.5, c='#e74c3c')
axes[2].axhline(0, color='grey', lw=0.5)
axes[2].set_xlabel('Production shortfall %')
axes[2].set_ylabel('Residual')
axes[2].set_title('Residuals vs production shortfall')

plt.tight_layout()
plt.show()

In [ ]:
results = pd.DataFrame([
    {'Model': 'PRIMARY — Annual USD return ~ shortfall_trail (trailing 5yr, clean)',
     'n': int(res_a3.nobs), 'R²': res_a3.rsquared,
     'shortfall coef': res_a3.params['shortfall_trail'],
     'shortfall p':    res_a3.pvalues['shortfall_trail']},
    {'Model': 'Annual USD return ~ shortfall_pct (centered 5yr, look-ahead upper bound)',
     'n': int(res_a3c.nobs), 'R²': res_a3c.rsquared,
     'shortfall coef': res_a3c.params['shortfall_pct'],
     'shortfall p':    res_a3c.pvalues['shortfall_pct']},
    {'Model': 'Annual USD return ~ prod_ret (log-return variant)',
     'n': int(res_a2.nobs), 'R²': res_a2.rsquared,
     'shortfall coef': res_a2.params['prod_ret'],
     'shortfall p':    res_a2.pvalues['prod_ret']},
    {'Model': 'Monthly USD no-lag ~ shortfall + seasonality (honest monthly R²)',
     'n': int(model_usd_simple.nobs), 'R²': model_usd_simple.rsquared,
     'shortfall coef': model_usd_simple.params['shortfall_pct'],
     'shortfall p':    model_usd_simple.pvalues['shortfall_pct']},
    {'Model': 'Monthly USD + lag_price (AR dominates)',
     'n': int(model_usd.nobs), 'R²': model_usd.rsquared,
     'shortfall coef': model_usd.params['log_prod'],
     'shortfall p':    model_usd.pvalues['log_prod']},
])
results = results.set_index('Model')
results['R²']             = results['R²'].round(3)
results['shortfall coef'] = results['shortfall coef'].round(4)
results['shortfall p']    = results['shortfall p'].round(3)
results

### Section 7 Conclusions

**Demand-side signals (Barry Callebaut, Hershey, Bunge, Nestlé, cocoa):**
- No candidate clears p < 0.10 once shortfall_trail is in the model.
- None reduce AIC — all add penalty without proportionate fit improvement.
- The one apparent signal (Ulker TRY, r≈0.54) is **endogenous**: Ulker equity returns are dominated by the same TRY depreciation episodes that make hazelnut USD prices look high in USD terms. It is not a demand-signal.
- Root cause: Turkey = ~70% global supply, buyer demand is inelastic (Ferrero, Nestlé have no near-term substitute for hazelnuts), and the largest buyer (Ferrero) is private with no tradeable equity.

**Frost features (ERA5 degree-hours):**
- April frost (april_dh) shows the strongest bivariate r (~0.29) but p ≈ 0.19 — not significant.
- **The reason is causal, not just statistical:** frost reduces production volume (the production channel). Production shortfall is already in the model, so frost adds no *independent* information about price. It is a leading indicator of shortfall, not an independent price driver.
- Implication for Component A (weather trigger): frost degree-hours are excellent for predicting *production* — that's the right model for the trigger side. Adding them to the price regression double-counts the same channel.

**Bottom line:** The single-factor model `ret_vwap_usd ~ shortfall_trail` is not only the cleanest specification — it is also the correct one given the market structure. Production shortfall is the price-formation mechanism; everything else (buyer equities, frost) either operates through that channel or is noise.

In [ ]:
# ERA5 frost features — test for direct price impact (vs. indirect via production)
frost = pd.read_csv(DATA / 'era5_frost_monthly.csv', index_col=0)
frost.index = frost.index.astype(int)
frost.index.name = 'crop_year'

frost_cols = [c for c in ['feb_dh', 'emarch_dh', 'lmarch_dh', 'april_dh', 'may_dh', 'frost_dh']
              if c in frost.columns]

fa = annual[['ret_vwap_usd', 'shortfall_trail']].join(frost[frost_cols], how='left')

frost_corr = []
for col in frost_cols:
    sub = fa[['ret_vwap_usd', col]].dropna()
    if len(sub) < 8:
        continue
    r, p = stats.pearsonr(sub['ret_vwap_usd'], sub[col])
    # Incremental: add frost to shortfall_trail baseline
    sub2 = fa[['ret_vwap_usd', 'shortfall_trail', col]].dropna()
    if len(sub2) >= 8:
        res_full = sm.OLS(sub2['ret_vwap_usd'],
                          sm.add_constant(sub2[['shortfall_trail', col]])).fit()
        res_base2 = sm.OLS(sub2['ret_vwap_usd'],
                           sm.add_constant(sub2[['shortfall_trail']])).fit()
        delta_r2 = res_full.rsquared - res_base2.rsquared
        frost_p  = res_full.pvalues[col]
    else:
        delta_r2 = np.nan; frost_p = np.nan
    frost_corr.append({'frost_feature': col, 'bivar_r': round(r, 3), 'bivar_p': round(p, 3),
                       'ΔR² vs baseline': round(delta_r2, 3), 'p in multivar': round(frost_p, 3),
                       'n': len(sub)})

frost_df = pd.DataFrame(frost_corr).sort_values('bivar_r', key=abs, ascending=False)
print('Frost feature correlations with annual USD price return:')
frost_df.set_index('frost_feature')

In [ ]:
# Multivariate: baseline (shortfall_trail only) vs each candidate addition
# Ulker excluded from this loop — it's TRY-denominated, endogenous with USD/TRY rate
candidate_features = [c for c in eq_cols if c != 'ulker_try'] + ['prod_ret_lag1']

baseline_sub = eq.dropna(subset=['ret_vwap_usd', 'shortfall_trail'])
res_base = sm.OLS(baseline_sub['ret_vwap_usd'],
                  sm.add_constant(baseline_sub[['shortfall_trail']])).fit()
base_aic = res_base.aic
base_r2  = res_base.rsquared
base_n   = int(res_base.nobs)

# Also add prod_ret_lag1 from annual panel (on/off proxy for biennial bearing)
eq['prod_ret_lag1'] = annual['prod_ret_lag1']

mv_rows = [{'feature': '(baseline: shortfall_trail only)',
            'R²': base_r2, 'ΔR²': 0.0, 'AIC': base_aic, 'feat_p': np.nan, 'n': base_n}]

for feat in candidate_features:
    sub = eq.dropna(subset=['ret_vwap_usd', 'shortfall_trail', feat])
    if len(sub) < 10:
        continue
    res = sm.OLS(sub['ret_vwap_usd'],
                 sm.add_constant(sub[['shortfall_trail', feat]])).fit()
    mv_rows.append({
        'feature': feat,
        'R²':    res.rsquared,
        'ΔR²':   res.rsquared - sm.OLS(sub['ret_vwap_usd'],
                                        sm.add_constant(sub[['shortfall_trail']])).fit().rsquared,
        'AIC':   res.aic,
        'feat_p': res.pvalues[feat],
        'n':     int(res.nobs),
    })

mv_df = pd.DataFrame(mv_rows)
mv_df['R²']     = mv_df['R²'].round(3)
mv_df['ΔR²']    = mv_df['ΔR²'].round(3)
mv_df['AIC']    = mv_df['AIC'].round(1)
mv_df['feat_p'] = mv_df['feat_p'].round(3)
print('Multivariate: baseline vs additions (all p-values > 0.05 — none improve AIC-adjusted fit)')
mv_df.set_index('feature')

In [ ]:
from scipy import stats

# Bivariate correlations of each equity return with hazelnut USD price return
target_col = 'ret_vwap_usd'
corr_rows = []
for col in basket_ret.columns:
    sub = eq[[target_col, col]].dropna()
    if len(sub) < 8:
        continue
    r, p = stats.pearsonr(sub[target_col], sub[col])
    corr_rows.append({'feature': col, 'r': r, 'p': p, 'n': len(sub)})

corr_df = pd.DataFrame(corr_rows).sort_values('r', key=abs, ascending=False).reset_index(drop=True)
corr_df['r']  = corr_df['r'].round(3)
corr_df['p']  = corr_df['p'].round(3)
corr_df['sig'] = corr_df['p'].apply(lambda p: '**' if p < 0.05 else ('*' if p < 0.10 else ''))
print('Bivariate correlations with annual hazelnut USD price return:')
corr_df

In [ ]:
# Load equity basket (annual prices indexed by harvest_year = crop_year)
basket = pd.read_csv(DATA / 'basket' / 'equity_basket_prices.csv', index_col=0)
basket.index = basket.index.astype(int)
basket.index.name = 'crop_year'

# Log-returns for each ticker
eq_cols = [c for c in basket.columns if c not in ('tur_usd', 'tryusd')]  # drop FX — endogenous
basket_ret = basket[eq_cols].apply(lambda x: np.log(x).diff())
basket_ret['tur_usd'] = np.log(basket['tur_usd']).diff()   # keep FX for reference only

# Merge with annual panel (match on crop_year)
eq = annual[['ret_vwap_usd', 'shortfall_trail']].join(basket_ret, how='left')

print(f'Equity basket: {len(basket_ret.columns)} tickers, {basket_ret.shape[0]} years')
print(f'Merged panel: {eq.shape}')
basket_ret.tail(5)

## 7. Demand-Side Signals and Frost Features

**Question:** Do equity/commodity prices of major buyers (Barry Callebaut, Hershey, Ulker, Bunge) or frost severity add explanatory power beyond production shortfall?

**Prior:** Turkey supplies ~70% of world hazelnuts. Major buyers (Ferrero, Nestlé, Barry Callebaut) have highly inelastic demand — they cannot easily substitute, so demand shocks propagate weakly into price. Ferrero is private (no tradeable equity). Ulker is a Turkey-domiciled firm whose equity returns are largely driven by TRY exchange rate — endogenous with our TRY/USD target.

**Method:** For each candidate feature, compute (a) bivariate r with `ret_vwap_usd`, and (b) ΔR² and AIC when added to the `shortfall_trail`-only baseline. Given n≈22, AIC penalises extra parameters heavily.